# IQM Pulla Tutorial

Circuit-to-pulse compilation and execution with IQM Pulla.

**Documentation:** https://docs.meetiqm.com/iqm-pulla/

In [ ]:
from __future__ import annotations
import os
import numpy as np
from qiskit import QuantumCircuit, visualization
from qiskit.compiler import transpile
from iqm.qiskit_iqm.iqm_transpilation import optimize_single_qubit_gates
from iqm.pulla.utils_qiskit import sweep_job_to_qiskit
from qaas.qpulla import qiskit_to_pulla


In [ ]:
import os
# os.environ["QPROVIDER_LOGLEVEL"] = "DEBUG"
os.environ["QPROVIDER_LOGLEVEL"] = "INFO"

## Setup Lexis Session

In [ ]:
from py4lexis.session import LexisSession

lexis_session = LexisSession()
print("Lexis session initialized")

In [ ]:
from qaas import QProvider, QBackend

token = lexis_session.get_access_token()
lexis_project = "vlq_demo_project"
resource_name = "qaas_user"

provider = QProvider(token, lexis_project, resource_name)

## Initialize Pulla and Backend

In [ ]:
# Get IQM server URL from backend or environment

# Create Pulla instance
p = provider.get_pulla()

print("Pulla initialized")
backend: QBackend = provider.get_backend()
print("Backend initialized")

In [ ]:

from qaas.qpulla import QPullaBackendIQM

client = provider.get_client()
dqa = client.get_dynamic_architecture()
compiler = p.get_standard_compiler()
pulla_backend: QPullaBackendIQM = QPullaBackendIQM(dqa,p,compiler)

## Create Bell State Circuit

In [ ]:
shots = 100

# Define quantum circuit
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

print("Original circuit:")
print(qc)
# qc.draw(output='mpl')

## Transpile Circuit

In [ ]:
# Transpile for IQM backend
qc_transpiled = backend.transpile(
    qc,
    layout_method='sabre',
    optimization_level=0# optimization_level=3
)
# Optimize single-qubit gates
qc_optimized = optimize_single_qubit_gates(qc_transpiled)

print("Transpiled and optimized circuit:")
print(qc_optimized)
# qc_optimized.draw(output="mpl")

## Convert to Pulla Format

In [ ]:
circuits, compiler = qiskit_to_pulla(p, pulla_backend, [qc_transpiled])

## Compile to Pulse Schedule

In [ ]:
# Compile circuit into instruction schedule playlist
# Create standard compiler
compiler = p.get_standard_compiler()
playlist, context = compiler.compile(circuits[0])
"""
AttributeError: 'QuantumCircuit' object has no attribute 'instructions'
"""

print(f"Compilation successful")
print(f"Context keys: {list(context.keys())}")

## Build Execution Settings

In [ ]:
# Build settings for execution
settings, context = compiler.build_settings(context, shots=shots)

print(f"Settings built for {shots} shots")
print(f"Settings type: {type(settings)}")

## Submit and Execute

In [ ]:
# Submit playlist returns SweepJob
job = p.submit_playlist(playlist, settings, context=context)

print(f"Job submitted")
print(f"Job ID: {job.job_id}")
print(f"Waiting for completion...")

# Wait for job completion
job.wait_for_completion()

print(f"Job completed")

## Process Results

In [ ]:
# Get raw results
raw_results = job.result()
print(f"Raw results:\n{raw_results}\n")

# Convert to Qiskit result format
qiskit_result = sweep_job_to_qiskit(
    job,
    shots=shots,
    execution_options=context['options']
)

# Get counts
counts = qiskit_result.get_counts()
print(f"Qiskit result counts:\n{counts}\n")

# Visualize
visualization.plot_histogram(counts)

## GHZ State Example

In [ ]:
# Create 3-qubit GHZ state
qc_ghz = QuantumCircuit(3, 3)
qc_ghz.h(0)
qc_ghz.cx(0, 1)
qc_ghz.cx(0, 2)
qc_ghz.measure_all()

print("GHZ circuit:")
print(qc_ghz)
qc_ghz.draw(output='mpl')

In [ ]:
# Transpile and optimize
qc_ghz_transpiled = backend.transpile(
    qc_ghz,
    layout_method='sabre',
    optimization_level=3
)
qc_ghz_optimized = optimize_single_qubit_gates(qc_ghz_transpiled)

# Convert to Pulla
circuits_ghz, compiler_ghz = qiskit_to_pulla(p, backend, qc_ghz_optimized)

# Compile
playlist_ghz, context_ghz = compiler_ghz.compile(circuits_ghz[0])
settings_ghz, context_ghz = compiler_ghz.build_settings(context_ghz, shots=shots)

# Execute
job_ghz = p.submit_playlist(playlist_ghz, settings_ghz, context=context_ghz)
job_ghz.wait_for_completion()

# Results
qiskit_result_ghz = sweep_job_to_qiskit(
    job_ghz,
    shots=shots,
    execution_options=context_ghz['options']
)

print(f"GHZ result counts:\n{qiskit_result_ghz.get_counts()}")
visualization.plot_histogram(qiskit_result_ghz.get_counts())

## Using Pulla Qiskit Backend

In [ ]:
from qaas.qpulla import QPullaBackendIQM
from iqm.iqm_client import DynamicQuantumArchitecture

# Get architecture
client = provider.get_client()
architecture: DynamicQuantumArchitecture = client.get_dynamic_architecture(None)

# Create standard compiler
compiler = p.get_standard_compiler()

# Create Pulla backend
pulla_backend = QPullaBackendIQM(architecture, p, compiler)

print(f"IQMPullaBackend created")
print(f"Backend name: {pulla_backend.name}")

In [ ]:
shots=100
# Use Pulla backend like regular Qiskit backend
qc_bell = QuantumCircuit(2, 2)
qc_bell.h(0)
qc_bell.cx(0, 1)
qc_bell.measure_all()

# Transpile for Pulla backend
qc_bell_transpiled = transpile(
    qc_bell,
    backend=pulla_backend,
    layout_method='sabre',
    optimization_level=3
)
qc_bell_optimized = optimize_single_qubit_gates(qc_bell_transpiled)

# Run on Pulla backend
job = pulla_backend.run(qc_bell_optimized, shots=shots)

print("Job executed via IQMPullaBackend")
print(f"Result counts: {job.result().get_counts()}")

visualization.plot_histogram(job.result().get_counts())

## Parameterized Circuit

In [ ]:
from qiskit.circuit import Parameter

# Create parameterized circuit
theta = Parameter('θ')

qc_param = QuantumCircuit(2, 2)
qc_param.ry(theta, 0)
qc_param.cx(0, 1)
qc_param.measure_all()

print(f"Parameterized circuit:")
print(qc_param)
print(f"Parameters: {qc_param.parameters}")

In [ ]:
# Execute with different parameter values
theta_values = [0, np.pi/4, np.pi/2, np.pi]

for theta_val in theta_values:
    # Bind parameter
    bound_qc = qc_param.assign_parameters({theta: theta_val})
    
    # Transpile
    transpiled = transpile(bound_qc, backend=pulla_backend, optimization_level=2)
    optimized = optimize_single_qubit_gates(transpiled)
    
    # Execute
    job = pulla_backend.run(optimized, shots=50)
    counts = job.result().get_counts()
    
    print(f"\nθ = {theta_val:.3f}:")
    print(f"  Counts: {counts}")

## Schedule Visualization

In [ ]:
from iqm.pulse.playlist.visualisation.base import inspect_playlist
from IPython.core.display import HTML

# Visualize the compiled playlist
# The playlist variable from earlier compilation
HTML(inspect_playlist(playlist, [0]))